# 1. Dataset Preparation

https://github.com/kerrgarr/SemanticSegmentationCityscapes

Download Cityscapes datasets: https://www.cityscapes-dataset.com/downloads

Datasets used: gtFine_trainvaltest.zip (241MB) and leftmg8bit_trainvaltest.zip (11GB).

Unzip both files in the data/ directory.

Your data/ folder structure should look like:

data/   
├──gtFine_trainvaltest/   
└──leftImg8bit_trainvaltest/    

In [ ]:
import os
import shutil

Create the dataset directory

In [ ]:
dataset_dir = './data/dataset/'
os.makedirs(dataset_dir, exist_ok=True)

Cityscape datasets paths

In [ ]:
gtFineDIR = './data/gtFine_trainvaltest/'
ImgDir = './data/leftImg8bit_trainvaltest/'

The Cityscapes test split does not contian image masks, only dummy blanks, so it cannot be used to evaluate model accuracy. 

We will use data from the original train split for testing, and exclude it from the actual trianing split used.

In [ ]:
# Train directory
gtFineDIR_train = os.path.join(gtFineDIR, 'gtFine/train')
ImgDIR_train = os.path.join(ImgDir, 'leftImg8bit/train')

dataset_train = os.path.join(dataset_dir,'train')

# Validation directory
gtFineDIR_valid = os.path.join(gtFineDIR, 'gtFine/val')
ImgDIR_valid = os.path.join(ImgDir, 'leftImg8bit/val')

dataset_valid = os.path.join(dataset_dir,'valid')

# Note the original test set is not used. 
# Test directory
gtFineDIR_test = gtFineDIR_train
ImgDIR_test = ImgDIR_train

dataset_test = os.path.join(dataset_dir, 'test')

Function to copy files

In [ ]:
def create_splits(sourcedir, targetdir, cities_list):
    extensions = ['labelIds', 'leftImg8bit']
    isExist = os.path.exists(targetdir)
    if not isExist:
        os.makedirs(targetdir, exist_ok=True)
    for city in cities_list:
        directory_city = os.path.join(sourcedir, city)
        image_files=os.listdir(directory_city)
        for image in image_files:
            if any(word in image for word in extensions):
                fullpath_image = os.path.join(directory_city, image)
                print(fullpath_image)
                try:
                    shutil.copy(fullpath_image, targetdir)
                except OSError as e:
                    print(e)

To prevent data leakage of similar frames from same cities, use different cities for different splits.

In [ ]:
train_cities = ['hamburg', 'cologne', 'bremen', 'strasbourg', 'zurich', 'ulm']

create_splits(gtFineDIR_train, dataset_train, train_cities)
create_splits(ImgDIR_train, dataset_train, train_cities)

In [ ]:
valid_cities = ['frankfurt', 'lindau']

create_splits(gtFineDIR_valid, dataset_valid, valid_cities)
create_splits(ImgDIR_valid, dataset_valid, valid_cities)

In [ ]:
# Note cities are from the original train split!
test_cities = ['stuttgart', 'jena']

create_splits(gtFineDIR_test, dataset_test, test_cities)
create_splits(ImgDIR_test, dataset_test, test_cities)

In [ ]:
# Roughly 70 / 15 / 15 split
for folder in os.listdir(dataset_dir):
    path = os.path.join(dataset_dir, folder)
    if os.path.isdir(path):
        print(folder, len(os.listdir(path))//2) # divide by 2 because each image has a matching mask file